In [1]:
from sklearn.model_selection import train_test_split
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
#--- Create the dataset ---# 

# 0:Met[Zsun],1:Age[Gyr],2:Rp[Rsun],3:Vinf[km/s],4:Mass1_i[MSUN],5:Mass2_i[Msun],6:R1[Rsun],7:R2[Rsun],8:Label,9:Mass1_f[MSUN],10:Mass2_f[MSUN],11:Sigma
data_dir = '/projects/b1091/SPH_ML_grid/machine_learning/data_n10k_splot22fplus_20251215.csv'

data_loaded  = np.loadtxt(data_dir, delimiter=',', dtype=float, skiprows = 1)

# Selecting only specific columns:
data = data_loaded[:, [1, 2, 3, 4, 5, 8]]
initial_masses = data_loaded[:, [4, 5]]
final_masses = data_loaded[:, [9, 10]]

# Re-assing final masses for merger cases to be all in mass1, otherwise there's a lot of confusion
merger_mask = ((data_loaded[:, 8] == 1) | (data_loaded[:, 8] == -1))
swap_rows = merger_mask & (final_masses[:, 0] == 0) & (final_masses[:, 1] > 0)
final_masses[swap_rows, 0], final_masses[swap_rows, 1] = (final_masses[swap_rows, 1], final_masses[swap_rows, 0])

# Use labels to startify the splits
labels = data[:,-1].astype(np.int64).copy()
labels[labels < 0] = 3 #replace locally the meaning of label -1

# Now create the regression dataset as [M1,f/Mtot,i , M2,f / Mtot,i, Mejec,f / Mtot,i]
y_data_reg = np.array([final_masses[:,0] / (initial_masses[:,0] + initial_masses[:,1]) , final_masses[:,1] / (initial_masses[:,0] + initial_masses[:,1]), ((initial_masses[:,0] + initial_masses[:,1]) - (final_masses[:,0] + final_masses[:,1]))/ (initial_masses[:,0] + initial_masses[:,1])] ).T

#Feature transform: log the x-data
x_data = data[:, :-1].copy()
x_data[:, 0] = np.log10(data[:, 0] + 0.001) # log ages
x_data[:, 1] = np.log10(data[:, 1] + 0.1) # log rp
x_data[:, 2] = np.log10(data[:, 2] + 10) # log vinfs
x_data[:, 3] = np.log(data[:, 3])   # ln M1,i
x_data[:, 4] = np.log(data[:, 4])   # ln M2,i

y_data = np.concatenate((labels[:, np.newaxis], y_data_reg), axis = 1)

print(np.shape(x_data), np.shape(y_data))

# Split into Training + Temporary (Validation + Test)
X_train, X_temp, y_train, y_temp, labels_train, labels_temp = train_test_split(x_data, y_data, labels, test_size=0.30, stratify = labels, random_state=42)

# Split the temporary into validation and testing sets 
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify = labels_temp, random_state=42)

print(np.shape(X_train), np.shape(X_val), np.shape(X_test))
print(np.shape(y_train), np.shape(y_val), np.shape(y_test))

# np.savez('data_splits_splot22f_1215.npz', X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test)

In [ ]:
# Checking relative representation in the dataset

y_train_labels = y_train[:,0]
y_val_labels   = y_val[:,0]
y_test_labels  = y_test[:,0]

for j in range(0,4):

    print("Class ", j)
    print(f"Number in Training Dataset: {len(y_train_labels[y_train_labels==j])} ({(len(y_train_labels[y_train_labels==j])/len(y_train_labels)) * 100})%")
    print(f"Number in Validation Dataset: {len(y_val_labels[y_val_labels==j])} ({(len(y_val_labels[y_val_labels==j])/len(y_val_labels)) * 100})%")
    print(f"Number in Testing Dataset: {len(y_test_labels[y_test_labels==j])} ({(len(y_test_labels[y_test_labels==j])/len(y_test_labels)) * 100})%")
